### Сравнение моделей (Model Comparison)

Эксперимент: обучить **7 моделей** на **6 наборах признаков** и сравнить метрики для одного resist_id.

**Табличные модели** (10 конфигураций) x **6 наборов** = 60 комбинаций + **4 per-series** конфигурации = **64 строки сводки**.

| № | Модель | Тип | Конфигурации |
|---|--------|------|----------------|
| 1 | LinearRegression | Табличная | 1 (baseline) |
| 2 | DecisionTreeRegressor | Табличная | 2 (shallow, deep) |
| 3 | CatBoostRegressor | Табличная | 3 (default, deep, conservative) |
| 4 | LGBMRegressor | Табличная | 2 (default, tuned) |
| 5 | XGBRegressor | Табличная | 2 (default, regularized) |
| 6 | ExponentialSmoothing (ETS) | Per-series | 2 (add, damped) |
| 7 | Prophet | Per-series | 2 (default, tuned) |

**Наборы признаков (Feature Sets):**

| Набор | Описание |
|-------|----------|
| A | Базовый: `op_code`, `month`, `year`, `fact_predict`, `workDayMonth_2`, `calculate_minute` |
| B | A + все `calc_*` из `calculate_parameter` |
| C | A + whitelist `calc_*` |
| A+lags | A + `lag_1`, `lag_12`, `rolling_mean_3` |
| B+lags | B + `lag_1`, `lag_12`, `rolling_mean_3` |
| C+lags | C + `lag_1`, `lag_12`, `rolling_mean_3` |

**Per-series** модели (ETS, Prophet) не используют признаки — только историю `fact_value`.

**Тест:** июль–декабрь 2025 (time-based split).  
**Метрики:** Точность, Стабильность, Качество (как в `build_metrics_dashboard_simple.py`).

In [ ]:
import json
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

HAS_PROPHET = True
try:
    from prophet import Prophet
    logging.getLogger("prophet").setLevel(logging.WARNING)
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
except ImportError:
    HAS_PROPHET = False
    print("prophet не установлен — конфигурации Prophet будут пропущены")

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)

#### Конфигурация

In [ ]:
RESIST_ID = 124

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

CALC_WHITELIST = [
    "calc_employeesFact",
    "calc_crossFunction",
    "calc_overtime",
    "calc_resistModel10",
    "calc_workDayMonth_2",
]

LAG_FEATURES = ["lag_1", "lag_12", "rolling_mean_3"]

#### Загрузка данных и подготовка (с calc_flat)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

df = df_model39[df_model39["resist_id"] == RESIST_ID].copy()
if df.empty:
    raise ValueError(
        f"Нет данных для resist_id={RESIST_ID}. "
        f"Доступные: {sorted(df_model39['resist_id'].dropna().unique()[:10].tolist())}..."
    )

df["op_code"] = df["operation_id"].astype("category").cat.codes
print(f"resist_id={RESIST_ID}: {df['operation_id'].nunique()} операций, {len(df)} строк")

#### Feature Engineering: lag_1, lag_12, rolling_mean_3

In [ ]:
group = df.groupby("operation_id")
df["lag_1"] = group["fact_value"].shift(1)
df["lag_12"] = group["fact_value"].shift(12)
df["rolling_mean_3"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)
df = df.infer_objects(copy=False)

print(f"Лаговые признаки добавлены: {LAG_FEATURES}")
df[["operation_id", "year", "month", "fact_value", "lag_1", "lag_12", "rolling_mean_3"]].head(15)

#### Проверка пропусков

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("Пропуски:")
    print(missing)
    df = df.fillna(0)
else:
    print("dataset не содержит пустых значений")

#### Наборы признаков (Feature Sets) и split

In [ ]:
BASE_FEATURES = ["op_code", "month", "year", "fact_predict", "workDayMonth_2", "calculate_minute"]

available_calc_cols = [c for c in df.columns if c.startswith("calc_") and c not in BASE_FEATURES]
available_whitelist = [c for c in CALC_WHITELIST if c in df.columns and c not in BASE_FEATURES]

FEATURE_SETS = {
    "A":      BASE_FEATURES,
    "B":      BASE_FEATURES + available_calc_cols,
    "C":      BASE_FEATURES + available_whitelist,
    "A+lags": BASE_FEATURES + LAG_FEATURES,
    "B+lags": BASE_FEATURES + available_calc_cols + LAG_FEATURES,
    "C+lags": BASE_FEATURES + available_whitelist + LAG_FEATURES,
}

for fs_name, fs_cols in FEATURE_SETS.items():
    print(f"  {fs_name:>6}: {len(fs_cols)} признаков")

mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
mask_train = ~mask_test

y_train = df.loc[mask_train, "fact_value"]
y_test = df.loc[mask_test, "fact_value"]

print(f"\nTrain: {mask_train.sum()} строк, Test: {mask_test.sum()} строк")
print(f"Табличных комбинаций: {len(FEATURE_SETS)} наборов x N конфигураций моделей")

#### Метрики + вспомогательные функции

In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo

#### Табличные модели: 10 конфигураций × 6 наборов признаков = 60 комбинаций

In [ ]:
from copy import deepcopy


def _build_model_configs(cat_idx: list[int]) -> dict:
    """Фабрика: каждый вызов создаёт свежие (не обученные) экземпляры моделей."""
    return {
        "LinearRegression": LinearRegression(),
        "DT-shallow": DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=42),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "CB-default": CatBoostRegressor(
            iterations=500, depth=6, learning_rate=0.1,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-deep": CatBoostRegressor(
            iterations=800, depth=8, learning_rate=0.05,
            verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "CB-conservative": CatBoostRegressor(
            iterations=300, depth=4, learning_rate=0.1,
            l2_leaf_reg=5, verbose=0, random_state=42, cat_features=cat_idx,
        ),
        "LGBM-default": LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbose=-1, random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            num_leaves=63, min_child_samples=10, verbose=-1, random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbosity=0, random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            reg_alpha=0.1, reg_lambda=5, verbosity=0, random_state=42,
        ),
    }


results = []
all_predictions = {}

t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)
print(f"fact_predict baseline: Точность={t_fp}% | Стабильность={s_fp}% | Качество={q_fp}%")

total_tabular = len(FEATURE_SETS) * len(_build_model_configs([0]))
combo_num = 0

for fs_name, fs_cols in FEATURE_SETS.items():
    cat_idx = [fs_cols.index("op_code")]
    model_configs = _build_model_configs(cat_idx)

    X_train_fs = df.loc[mask_train, fs_cols]
    X_test_fs = df.loc[mask_test, fs_cols]

    print(f"\n{'='*90}")
    print(f"Набор: {fs_name} ({len(fs_cols)} признаков)")
    print(f"{'='*90}")
    print(f"{'Config':<22} | {'Accuracy':>8} | {'Stability':>9} | {'Quality':>7} | {'Time':>7}")
    print(f"{'-'*90}")

    for config_name, model in model_configs.items():
        combo_num += 1
        combo_key = f"{config_name} | {fs_name}"
        t0 = time.perf_counter()
        model.fit(X_train_fs, y_train)
        y_pred = np.maximum(model.predict(X_test_fs), 0)
        elapsed = round(time.perf_counter() - t0, 2)

        tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
        results.append({
            "Конфигурация": config_name,
            "Набор признаков": fs_name,
            "Тип": "табличная",
            "Точность, %": tochnost,
            "Стабильность, %": stab,
            "Качество, %": kachestvo,
            "Время, сек": elapsed,
            "Точн. fact_predict, %": t_fp,
            "Стаб. fact_predict, %": s_fp,
            "Кач. fact_predict, %": q_fp,
        })
        all_predictions[combo_key] = y_pred
        print(f"{config_name:<22} | {tochnost:7.2f}% | {stab:8.2f}% | {kachestvo:6.2f}% | {elapsed:6.2f}s")

print(f"\nТабличные комбинации завершены: {combo_num}/{total_tabular}")

#### Per-series модели: ETS и Prophet (4 конфигурации)

In [ ]:
def train_ets_per_series(df_full, mask_train, mask_test, trend="add", damped_trend=False):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 4:
                raise ValueError("too short")
            model = ExponentialSmoothing(
                y_hist, trend=trend, damped_trend=damped_trend, seasonal=None,
            )
            fit = model.fit(optimized=True, use_brute=False)
            fc = fit.forecast(n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_prophet_per_series(df_full, mask_train, mask_test, prophet_params):
    if not HAS_PROPHET:
        return None
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 4:
                raise ValueError("too short")
            prophet_df = pd.DataFrame({
                "ds": pd.to_datetime(train_part["date"]),
                "y": y_hist,
            })
            m = Prophet(**prophet_params)
            m.fit(prophet_df)
            future = m.make_future_dataframe(periods=n_fc, freq="MS")
            fc = m.predict(future)["yhat"].iloc[-n_fc:].values
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


SERIES_CONFIGS = {
    "ETS-add": {"fn": "ets", "trend": "add", "damped_trend": False},
    "ETS-damped": {"fn": "ets", "trend": "add", "damped_trend": True},
    "Prophet-default": {
        "fn": "prophet",
        "params": {"seasonality_mode": "additive", "n_changepoints": 10,
                   "yearly_seasonality": False, "weekly_seasonality": False,
                   "daily_seasonality": False},
    },
    "Prophet-tuned": {
        "fn": "prophet",
        "params": {"seasonality_mode": "multiplicative", "n_changepoints": 5,
                   "changepoint_prior_scale": 0.1,
                   "yearly_seasonality": False, "weekly_seasonality": False,
                   "daily_seasonality": False},
    },
}

print(f"\n{'='*80}")
print(f"Per-series модели ({len(SERIES_CONFIGS)} конфигураций)")
print(f"{'='*80}")

for config_name, cfg in SERIES_CONFIGS.items():
    t0 = time.perf_counter()

    if cfg["fn"] == "ets":
        y_pred = train_ets_per_series(
            df, mask_train, mask_test,
            trend=cfg["trend"], damped_trend=cfg["damped_trend"],
        )
    elif cfg["fn"] == "prophet":
        y_pred = train_prophet_per_series(df, mask_train, mask_test, cfg["params"])
        if y_pred is None:
            print(f"{config_name:<22} | пропущен (prophet не установлен)")
            continue
    else:
        continue

    elapsed = round(time.perf_counter() - t0, 2)
    tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)

    results.append({
        "Конфигурация": config_name,
        "Набор признаков": "—",
        "Тип": "per-series",
        "Точность, %": tochnost,
        "Стабильность, %": stab,
        "Качество, %": kachestvo,
        "Время, сек": elapsed,
        "Точн. fact_predict, %": t_fp,
        "Стаб. fact_predict, %": s_fp,
        "Кач. fact_predict, %": q_fp,
    })
    all_predictions[config_name] = y_pred
    print(f"{config_name:<22} | {tochnost:7.2f}% | {stab:8.2f}% | {kachestvo:6.2f}% | {elapsed:6.2f}s")

print(f"\nВсего комбинаций: {len(results)} (табличных: {combo_num}, per-series: {len(results) - combo_num})")

#### Сводная таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results["Комбинация"] = df_results.apply(
    lambda r: f"{r['Конфигурация']} | {r['Набор признаков']}" if r["Набор признаков"] != "—" else r["Конфигурация"],
    axis=1,
)
df_results = df_results.sort_values("Качество, %", ascending=False).reset_index(drop=True)
print(f"Всего строк в сводке: {len(df_results)}")
df_results.head(20)

In [ ]:
TOP_N = 20
df_top = df_results.head(TOP_N)

fig, axes = plt.subplots(1, 3, figsize=(18, max(8, TOP_N * 0.35)))
metrics_to_plot = ["Точность, %", "Стабильность, %", "Качество, %"]
for ax, metric in zip(axes, metrics_to_plot):
    df_plot = df_top.sort_values(metric, ascending=True)
    colors = ["#2196F3" if t == "табличная" else "#FF9800" for t in df_plot["Тип"]]
    ax.barh(df_plot["Комбинация"], df_plot[metric], color=colors)
    ax.set_xlabel(metric)
    ax.set_title(metric)
plt.suptitle(f"TOP-{TOP_N} комбинаций (resist_id={RESIST_ID})", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nЛучший набор по каждой модели:")
for cfg_name in df_results["Конфигурация"].unique():
    best_row = df_results[df_results["Конфигурация"] == cfg_name].iloc[0]
    print(f"  {cfg_name:<22} → {best_row['Набор признаков']:<7} | Качество={best_row['Качество, %']:.2f}%")

#### Лучшая конфигурация: детали по операциям

In [ ]:
best_row = df_results.iloc[0]
best_combo = best_row["Комбинация"]
best_config = best_row["Конфигурация"]
best_fs = best_row["Набор признаков"]
best_type = best_row["Тип"]
print(f"Лучшая комбинация по Качеству: {best_combo} ({best_type})")
print(f"  Модель: {best_config}, Набор: {best_fs}")
print(f"  Точность={best_row['Точность, %']}% | Стабильность={best_row['Стабильность, %']}% | Качество={best_row['Качество, %']}%")

best_pred = all_predictions[best_combo]
df_test = df.loc[mask_test].copy()
df_test["predict_new"] = best_pred

rows_op = []
for (rid, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
    t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
    t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_new"].values)
    rows_op.append({
        "resist_id": rid,
        "operation_id": op_id,
        "Точность (fact_predict)": t1,
        "Стабильность (fact_predict)": s1,
        "Качество (fact_predict)": q1,
        f"Точность ({best_combo})": t2,
        f"Стабильность ({best_combo})": s2,
        f"Качество ({best_combo})": q2,
    })

df_metrics_op = pd.DataFrame(rows_op)
print(f"Операций в тесте: {len(df_metrics_op)}")
df_metrics_op.head(10)

#### Экспорт в Excel

In [ ]:
df_detail = df_test[["resist_id", "operation_id", "year", "month",
                      "fact_value", "fact_predict", "predict_new"]].copy()
df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
df_detail = df_detail.rename(columns={
    "fact_value": "fact_value (факт)",
    "fact_predict": "fact_predict (текущий)",
    "predict_new": f"{best_combo} (новый)",
})

out_path = Path("../data/processed/model_comparison.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    df_results.to_excel(w, sheet_name="Сводка моделей", index=False)
    df_detail.to_excel(w, sheet_name="Данные теста", index=False)
    df_metrics_op.to_excel(w, sheet_name="По операциям", index=False)

print(f"Excel сохранён: {out_path.resolve()}")
print(f"Лист 'Сводка моделей': {len(df_results)} строк (60 табличных + до 4 per-series)")
print(f"Лист 'Данные теста': {len(df_detail)} строк (прогноз лучшей комбинации: {best_combo})")

#### Выводы

* Проведён полный перебор: **10 конфигураций моделей × 6 наборов признаков = 60 табличных комбинаций** + **4 per-series** = **64 строки сводки**.
* Наборы: **A** (базовый), **B** (+ все calc_*), **C** (+ whitelist calc_*), **A+lags**, **B+lags**, **C+lags**.
* **Лучшая комбинация** (модель + набор) определена по метрике **Качество** и экспортирована в Excel.
* **Per-series модели** (ETS, Prophet) не используют наборы признаков — обучаются на каждой операции отдельно.
* Результаты в `model_comparison.xlsx`: Сводка моделей (64 строки), Данные теста, По операциям.